# Assignment 4: Pricing Agent Evaluation Framework

## Objective
Build a comprehensive evaluation framework for the tool-enhanced pricing agent from, measuring both prompt effectiveness and tool utilization.

## Requirements
**Evaluation Components:**
- Prompt evaluation with binary metrics (accuracy, completeness, clarity)
- Tool usage metrics (precision, recall, efficiency)
- Performance comparison across agent versions
- Cost and execution time analysis


## Setup & Dependencies

Install and import required packages for the evaluation framework.

In [ ]:
# Install required packages
!pip install -q langchain langchain-groq langchain-community
!pip install -q pandas numpy matplotlib
!pip install -q pydantic

In [1]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import os
import getpass
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re
import time
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field


In [2]:
# Set up your Groq API key
print("Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key


print("API key configured!")

Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
API key configured!


## Part 1: Import Pricing Agent Tools

First, import or recreate the pricing tools from Assignment 3.

In [77]:
@tool
def margin_calculator(cost_price: float, target_margin: float) -> dict:
    """
    Calculate optimal selling price based on cost and target margin.
    
    Args:
        cost_price (float): The cost to produce/acquire the product
        target_margin (float): Desired margin as decimal (e.g., 0.25 for 25%)
    
    Returns:
        dict: Contains selling_price, margin_dollar, and validation info
    """
    price = cost_price / (1 - target_margin)
    margin_dollar = price - cost_price
    validation = {
        "valid": True,
        "message": ""
    }
    if target_margin >= 1.0:
        validation["valid"] = False
        validation["message"] = "Target margin must be less than 1.0"
    elif target_margin < 0:
        validation["valid"] = False
        validation["message"] = "Target margin must be non-negative"

    return {
        "selling_price": price,
        "margin_dollar": margin_dollar,
        "validation": validation
    }

@tool
def elasticity_adjustment(base_price: float, elasticity: str, price_change_percent: float) -> dict:
    """
    Adjust pricing based on demand elasticity to optimize revenue.
    
    Args:
        base_price (float): Starting price point
        elasticity (str): Elasticity level - 'low', 'medium', 'medium-high', or 'high'
        price_change_percent (float): Proposed price change as percentage (e.g., 10 for 10% increase)
    
    Returns:
        dict: Contains adjusted_price, demand_impact, revenue_impact
    """
    elasticity_coefficients = {
        'low': -0.5,
        'medium': -1.0,
        'medium-high': -1.5,
        'high': -2.0
    }
    demand_change = elasticity_coefficients.get(elasticity, 0) * (price_change_percent / 100)
    adjusted_price = base_price * (1 + price_change_percent / 100)
    revenue_impact = (1 + price_change_percent / 100) * (1 + demand_change) - 1
    
    return {
        "adjusted_price": adjusted_price,
        "demand_impact": demand_change,
        "revenue_impact": revenue_impact
    }

@tool
def competitor_matching(our_cost: float, competitor_price: float, positioning: str = "match") -> dict:
    """
    Calculate optimal pricing relative to competitor prices.
    
    Args:
        our_cost (float): Our cost to produce the product
        competitor_price (float): Competitor's selling price
        positioning (str): Strategy - 'undercut', 'match', 'premium', or 'aggressive'
    
    Returns:
        dict: Contains recommended_price, our_margin, competitive_analysis
    """
    
    positioning_strategies = {
        'aggressive': 0.85,
        'undercut': 0.92,
        'match': 0.98,
        'premium': 1.10
    }

    multiplier = positioning_strategies.get(positioning, 0.98)
    recommended_price = competitor_price * multiplier
    our_margin = recommended_price - our_cost
    competitive_analysis = {
        "positioning": positioning,
        "multiplier": multiplier,
        "recommended_price": recommended_price,
        "our_margin": our_margin,
        "covers_cost": recommended_price > our_cost
    }   
    
    return {
        "recommended_price": recommended_price,
        "our_margin": our_margin,
        "competitive_analysis": competitive_analysis
    }

print("Testing Pricing Tools:")
print("="*40)

margin_result = margin_calculator.invoke({"cost_price": 400, "target_margin": 0.25})
print(f"Margin test: {margin_result}")

elasticity_result = elasticity_adjustment.invoke({"base_price": 533, "elasticity": "medium", "price_change_percent": 8.5})
print(f"Elasticity test: {elasticity_result}")

competitor_result = competitor_matching.invoke({"our_cost": 400, "competitor_price": 579, "positioning": "match"})
print(f"Competitor test: {competitor_result}")

print("Tools implementation pending...")

# List of pricing tools
pricing_tools = [margin_calculator, elasticity_adjustment, competitor_matching]
print(f"TODO: Import {len(pricing_tools)} pricing tools from Assignment 3")

Testing Pricing Tools:
Margin test: {'selling_price': 533.3333333333334, 'margin_dollar': 133.33333333333337, 'validation': {'valid': True, 'message': ''}}
Elasticity test: {'adjusted_price': 578.305, 'demand_impact': -0.085, 'revenue_impact': -0.007225000000000037}
Competitor test: {'recommended_price': 567.42, 'our_margin': 167.41999999999996, 'competitive_analysis': {'positioning': 'match', 'multiplier': 0.98, 'recommended_price': 567.42, 'our_margin': 167.41999999999996, 'covers_cost': True}}
Tools implementation pending...
TODO: Import 3 pricing tools from Assignment 3


## Part 2: Create Multiple Agent Versions

Create different agent versions with varying prompt strategies to evaluate.

In [78]:
class PricingAgent:
    def __init__(self, prompt_version: str):
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1, max_tokens=2048)
        
        self.tools = pricing_tools
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        
        self.prompt_version = prompt_version
        self.system_message = self._get_system_prompt(prompt_version)
        
    def _get_system_prompt(self, version: str) -> SystemMessage:
        tool_rule = (
            "\n\nIMPORTANT: Call tools ONE AT A TIME with concrete numeric values only. "
            "Never nest function calls or reference tool outputs inside tool arguments. "
            "After each tool returns a result, use that result's value in subsequent tool calls."
        )
        prompts = {
            "basic": """
            You are a pricing optimization agent. 
            Your task is to analyze the provided query and determine the optimal price for a product using the available tools. 
            Focus on accuracy and business impact in your analysis.    
            """ + tool_rule,
            
            "structured": """
            You are a pricing optimization agent. 
            Your task is to analyze the provided query and determine the optimal price for a product using the available tools.
            The process should be structured as follows:
            1. Understand the query and identify relevant factors (cost, margin, elasticity, competition).
            2. Select appropriate tools based on the query context.
            3. Execute tools step-by-step, ensuring to interpret results at each stage.
            4. Synthesize tool outputs to arrive at a final pricing recommendation.
            """ + tool_rule,
            
            "comprehensive": """
            You are a highly sophisticated pricing optimization agent with expertise in economics, market analysis, and competitive strategy.
            Your task is to analyze the provided query and determine the optimal price for a product using the available tools.

            The tools at your disposal include:
            - Margin Calculator: Calculates optimal selling price based on cost and target margin.
            - Elasticity Adjustment: Adjusts pricing based on demand elasticity to optimize revenue.
            - Competitor Matching: Calculates optimal pricing relative to competitor prices.
            The process should be comprehensive and consider multiple factors, including market trends, competitor pricing, and customer behavior.

            You should follow a multi-step analysis process:
            1. Thoroughly understand the query and identify all relevant factors (cost, margin, elasticity, competition).
            2. Select and execute appropriate tools based on the query context, ensuring to interpret results at each stage.
            3. Synthesize tool outputs to arrive at a final pricing recommendation, considering business impact and accuracy in your analysis.

            Business impact considerations should include:
            - Revenue implications of the recommended price
            - Competitive positioning in the market
            - Customer perception and demand sensitivity
            """ + tool_rule
        }
        return SystemMessage(content=prompts.get(version, prompts["basic"]))
    
    def calculate_price(self, query: str) -> dict:
        """Calculate price based on query and return response with metadata"""

        start_time = time.time()

        messages = [self.system_message, HumanMessage(content=query)]
        response = self.llm_with_tools.invoke(messages)
        messages.append(response)  # ← append AI response BEFORE tool results

        tools_list = []
        max_iterations = 6  # guard against infinite loops
        iteration = 0
        while response.tool_calls and iteration < max_iterations:
            iteration += 1
            for tool_call in response.tool_calls:
                tool_fn = next((t for t in self.tools if t.name == tool_call["name"]), None)
                if tool_fn:
                    try:
                        tool_result = tool_fn.invoke(tool_call["args"])
                    except Exception as e:
                        tool_result = {"error": str(e)}
                    messages.append(ToolMessage(
                        content=str(tool_result),
                        tool_call_id=tool_call["id"]
                    ))
                    tools_list.append({"tool": tool_fn.name, "result": str(tool_result)})
            response = self.llm_with_tools.invoke(messages)
            messages.append(response)  # ← append next AI response too

        execution_time = time.time() - start_time
        
        return {
            "response": response.content,
            "tool_calls": tools_list,
            "execution_time": execution_time,
            "prompt_version": self.prompt_version
        }

agents = {
    "basic": PricingAgent("basic"),
    "structured": PricingAgent("structured"),
    "comprehensive": PricingAgent("comprehensive")
}

print("Created 3 agent versions: basic, structured, comprehensive")


Created 3 agent versions: basic, structured, comprehensive


## Part 3: Define Evaluation Dataset

Create test scenarios to evaluate the agents.

In [79]:
evaluation_dataset = [
    {
        "id": 1,
        "query": "Calculate the selling price for a product with cost $100 and target margin 25%",
        "expected_tools": ["margin_calculator"],
        "expected_price_range": (130, 150),  # $100 / (1 - 0.25) = $133.33; widened to allow for minor adjustments
        "category": "single_tool",
        "key_factors": ["margin", "cost"]
    },
    # {
    #     "id": 2,
    #     "query": "Adjust the price of a product with base price $200 based on medium elasticity and a proposed price increase of 10%",
    #     "expected_tools": ["elasticity_adjustment"],
    #     "expected_price_range": (210, 220),  # $200 * (1 + 0.10) = $220
    #     "category": "single_tool",
    #     "key_factors": ["elasticity", "price_change"]
    # },
    # {
    #     "id": 3,
    #     "query": "Calculate the optimal price for a product with cost $400, competitor price $579, and a strategy to match the competitor",
    #     "expected_tools": ["competitor_matching"],
    #     "expected_price_range": (560, 570),  # $579 * 0.98 = $567.42
    #     "category": "single_tool",
    #     "key_factors": ["competition", "cost"]
    # },
    # {
    #     "id": 4,
    #     "query": "For a product with cost $150, target margin 30%, and medium elasticity, calculate the optimal price considering both margin and elasticity adjustments.",
    #     "expected_tools": ["margin_calculator", "elasticity_adjustment"],
    #     "expected_price_range": (190, 210),  # Margin price = $150 / (1 - 0.30) = $214.29, Elasticity adjustment may reduce it
    #     "category": "multi_tool",
    #     "key_factors": ["margin", "elasticity", "cost"]
    # },
    # {
    #     "id": 5,
    #     "query": "Determine the optimal price for a product with cost $300, competitor price $350, and a strategy to undercut the competitor while maintaining a target margin of 20%. Consider both competitive positioning and margin requirements.",
    #     "expected_tools": ["competitor_matching", "margin_calculator"],
    #     "expected_price_range": (280, 320),  # Undercut price = $350 * 0.92 = $322, Margin price = $300 / (1 - 0.20) = $375
    #     "category": "multi_tool",
    #     "key_factors": ["competition", "margin", "cost"]
    # },
    # {
    #     "id": 6,
    #     "query": "Calculate the optimal price for a product with cost $250, target margin 25%, and high elasticity. Consider how the price change will impact demand and revenue.",
    #     "expected_tools": ["margin_calculator", "elasticity_adjustment"],
    #     "expected_price_range": (300, 350),  # Margin price = $250 / (1 - 0.25) = $333.33, Elasticity adjustment may reduce it significantly
    #     "category": "multi_tool",
    #     "key_factors": ["margin", "elasticity", "cost"]
    # },
    # {
    #     "id": 7,
    #     "query": "Determine the optimal price for a product with cost $500, competitor price $550, and a strategy to position as premium. Consider both competitive positioning and margin requirements.",
    #     "expected_tools": ["competitor_matching", "margin_calculator"],
    #     "expected_price_range": (550, 600),  # Premium price = $550 * 1.10 = $605, Margin price = $500 / (1 - 0.20) = $625
    #     "category": "multi_tool",
    #     "key_factors": ["competition", "margin", "cost"]
    # },
    # {
    #     "id": 8,
    #     "query": "Calculate the optimal price for a product with cost $80, target margin 15%, and low elasticity. Consider how the price change will impact demand and revenue.",
    #     "expected_tools": ["margin_calculator", "elasticity_adjustment"],
    #     "expected_price_range": (90, 100),  # Margin price = $80 / (1 - 0.15) = $94.12, Elasticity adjustment may have minimal impact
    #     "category": "multi_tool",
    #     "key_factors": ["margin", "elasticity", "cost"]
    # },
    # {
    #     "id": 9,
    #     "query": "Determine the optimal price for a product with cost $120, competitor price $150, and a strategy to match the competitor. Consider both competitive positioning and margin requirements.",
    #     "expected_tools": ["competitor_matching", "margin_calculator"],
    #     "expected_price_range": (140, 160),  # Match price = $150 * 0.98 = $147, Margin price = $120 / (1 - 0.20) = $150
    #     "category": "multi_tool",
    #     "key_factors": ["competition", "margin", "cost"]
    # },
    # {
    #     "id": 10,
    #     "query": "Calculate the optimal price for a product with cost $60, target margin 20%, and medium-high elasticity. Consider how the price change will impact demand and revenue.",
    #     "expected_tools": ["margin_calculator", "elasticity_adjustment"],
    #     "expected_price_range": (70, 80),  # Margin price = $60 / (1 - 0.20) = $75, Elasticity adjustment may reduce it
    #     "category": "multi_tool",
    #     "key_factors": ["margin", "elasticity", "cost"]
    # }
]

print(f"Current scenarios: {len(evaluation_dataset)}")


Current scenarios: 1


## Part 4: Prompt Evaluation Metrics

Define binary metrics to evaluate prompt effectiveness.

In [80]:
def evaluate_prompt_response(query: str, response: str, expected_price_range: tuple, 
                            key_factors: list, tool_calls: list) -> dict:
    """Evaluate response using binary metrics (0 or 1)"""
    
    evaluator_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    eval_prompt = f"""
    You are an expert evaluator assessing the quality of a pricing recommendation response.
    Evaluate ONLY based on the criteria below. Do NOT apply your own business or mathematical knowledge.

    Criteria:
    - Accuracy: Does the response contain a FINAL recommended price that falls within the expected range?
      IMPORTANT: The range is intentionally wide to allow for multi-step reasoning. 
      Check ONLY whether the last/final price mentioned in the response is between ${expected_price_range[0]} and ${expected_price_range[1]} (inclusive).
      Do NOT judge whether the calculation is mathematically optimal.
    - Completeness: Are all key factors listed below explicitly mentioned anywhere in the response?
      Key Factors: {', '.join(key_factors)}
    - Clarity: Is the explanation clear and understandable for business users?
    
    Query: {query}
    Response: {response}
    Expected Range: ${expected_price_range[0]} - ${expected_price_range[1]}
    
    Respond ONLY with exactly this format (use 0 or 1 for each):
    Accuracy: X
    Completeness: X
    Clarity: X
    """
    
    eval_response = evaluator_llm.invoke([HumanMessage(content=eval_prompt)])
    response_text = eval_response.content
    
    # Parse binary scores using regex
    def extract_score(metric: str, text: str) -> int:
        match = re.search(rf"{metric}:\s*([01])", text, re.IGNORECASE)
        return int(match.group(1)) if match else 0
    
    scores = {
        "accuracy":     extract_score("Accuracy", response_text),
        "completeness": extract_score("Completeness", response_text),
        "clarity":      extract_score("Clarity", response_text),
        "price_in_range": 0,
        "extracted_price": None
    }
    
    # Extract price from agent response and check if it falls within expected range
    price_match = re.search(r'\$([0-9,]+\.?[0-9]*)', response)
    if price_match:
        extracted_price = float(price_match.group(1).replace(",", ""))
        scores["extracted_price"] = extracted_price
        scores["price_in_range"] = 1 if expected_price_range[0] <= extracted_price <= expected_price_range[1] else 0
    
    return scores


## Part 5: Tool Evaluation Metrics

Define metrics to evaluate tool usage effectiveness.

In [81]:
@dataclass
class ToolEvaluation:
    """Tool usage evaluation for a single test case"""
    query: str
    expected_tools: List[str]
    actual_tools: List[str]
    tool_precision: float
    tool_recall: float
    tool_success_rate: float
    execution_time: float
    tool_count: int
    optimal_count: int
    
    @property
    def efficiency_classification(self) -> str:
        if self.tool_count < self.optimal_count:
            return "too_few"
        elif self.tool_count == self.optimal_count:
            return "optimal"
        else:            
            return "too_many"

def evaluate_tool_usage(expected_tools: List[str], tool_calls: List[dict]) -> ToolEvaluation:
    """Evaluate tool usage metrics"""
    
    actual_tools = [call['tool'] for call in tool_calls]

    correct_tools = set(expected_tools) & set(actual_tools)
    
    precision = len(correct_tools) / len(actual_tools) if actual_tools else 0.0
    
    recall = len(correct_tools) / len(expected_tools) if expected_tools else 0.0
    
    successful_calls = sum(1 for call in tool_calls if 'error' not in call.get('result', ''))
    success_rate = successful_calls / len(tool_calls) if tool_calls else 0.0
    
    return ToolEvaluation(
        query="",
        expected_tools=expected_tools,
        actual_tools=actual_tools,
        tool_precision=precision,
        tool_recall=recall,
        tool_success_rate=success_rate,
        execution_time=0,
        tool_count=len(actual_tools),
        optimal_count=len(expected_tools)
    )

## Part 6: Run Comprehensive Evaluation

Execute the evaluation across all agents and scenarios.

In [82]:
def run_comprehensive_evaluation(agents: dict, dataset: list) -> pd.DataFrame:
    """Run evaluation for all agents on all test cases"""
    
    results = []
    
    for agent_name, agent in agents.items():
        for scenario in dataset:
            print(f"Evaluating {agent_name} on scenario {scenario['id']}...", flush=True)
            try:
                response_data = agent.calculate_price(scenario['query'])
                prompt_scores = evaluate_prompt_response(
                    query=scenario['query'],
                    response=response_data['response'],
                    expected_price_range=scenario['expected_price_range'],
                    key_factors=scenario['key_factors'],
                    tool_calls=response_data['tool_calls']
                )
                tool_eval = evaluate_tool_usage(
                    expected_tools=scenario['expected_tools'],
                    tool_calls=response_data['tool_calls']
                )
                result_entry = {
                    'agent': agent_name,
                    'test_id': scenario['id'],
                    'category': scenario['category'],
                    'accuracy': prompt_scores['accuracy'],  # LLM judge (range-only check)
                    'completeness': prompt_scores['completeness'],
                    'clarity': prompt_scores['clarity'],
                    'price_in_range': prompt_scores['price_in_range'],
                    'extracted_price': prompt_scores['extracted_price'],
                    'tool_precision': tool_eval.tool_precision,
                    'tool_recall': tool_eval.tool_recall,
                    'tool_success_rate': tool_eval.tool_success_rate,
                    'execution_time': response_data['execution_time'],
                    'tool_count': tool_eval.tool_count,
                    'optimal_tool_count': tool_eval.optimal_count,
                    'efficiency_classification': tool_eval.efficiency_classification
                }
                results.append(result_entry)
                print(f"  ✓ Done (time={response_data['execution_time']:.1f}s, tools={tool_eval.tool_count})", flush=True)
            except Exception as e:
                print(f"  ✗ Error on {agent_name}/scenario {scenario['id']}: {e}", flush=True)
    
    return pd.DataFrame(results) if results else pd.DataFrame()

# Run the evaluation
print("Starting comprehensive evaluation...\n", flush=True)
evaluation_results = run_comprehensive_evaluation(agents, evaluation_dataset)


Starting comprehensive evaluation...

Evaluating basic on scenario 1...
  ✓ Done (time=1.8s, tools=3)
Evaluating structured on scenario 1...
  ✓ Done (time=26.2s, tools=3)
Evaluating comprehensive on scenario 1...
  ✓ Done (time=24.2s, tools=1)


## Part 7: Aggregate Results

Calculate aggregate metrics for each agent.

In [83]:
agent_summary = evaluation_results.groupby('agent').agg({
    'accuracy': 'mean',
    'completeness': 'mean',
    'clarity': 'mean',
    'tool_precision': 'mean',
    'tool_recall': 'mean',
    'execution_time': 'mean'
})

prompt_score = (agent_summary['accuracy'] + agent_summary['completeness'] + agent_summary['clarity']) / 3
tool_score = (agent_summary['tool_precision'] + agent_summary['tool_recall']) / 2
overall_score = (prompt_score + tool_score) / 2

print("\nAgent Performance Summary:")
print(agent_summary)
print("\nComposite Scores:")
for agent in agent_summary.index:
    print(f"{agent}: Prompt Score={prompt_score[agent]:.2f}, Tool Score={tool_score[agent]:.2f}, Overall Score={overall_score[agent]:.2f}")


Agent Performance Summary:
               accuracy  completeness  clarity  tool_precision  tool_recall  \
agent                                                                         
basic               1.0           1.0      1.0        0.333333          1.0   
comprehensive       1.0           0.0      1.0        1.000000          1.0   
structured          1.0           1.0      1.0        0.333333          1.0   

               execution_time  
agent                          
basic                1.800489  
comprehensive       24.154154  
structured          26.214669  

Composite Scores:
basic: Prompt Score=1.00, Tool Score=0.67, Overall Score=0.83
comprehensive: Prompt Score=0.67, Tool Score=1.00, Overall Score=0.83
structured: Prompt Score=1.00, Tool Score=0.67, Overall Score=0.83


## Part 8: Cost Analysis

Estimate and analyze costs for each agent configuration.

In [84]:
def estimate_cost(agent_results):
    """Estimate cost based on token usage"""

    TOKENS_PER_CHAR = 4
    COST_PER_1K_INPUT = 0.0005
    COST_PER_1K_OUTPUT = 0.001
    COST_PER_TOOL_CALL = 0.001

    costs = []

    for _, row in agent_results.iterrows():
        # 1. Estimate input/output tokens
        # Base: system prompt (~300 tokens) + user query (~80 tokens)
        # Each tool call adds ~150 input tokens (schema + result) and ~60 output tokens
        base_input_tokens = 380
        base_output_tokens = 200
        tool_input_tokens = row['tool_count'] * 150
        tool_output_tokens = row['tool_count'] * 60

        total_input = base_input_tokens + tool_input_tokens
        total_output = base_output_tokens + tool_output_tokens

        # 2. Calculate tool costs
        tool_cost = row['tool_count'] * COST_PER_TOOL_CALL

        # 3. Sum total cost per task
        input_cost = (total_input / 1000) * COST_PER_1K_INPUT
        output_cost = (total_output / 1000) * COST_PER_1K_OUTPUT
        total_cost = input_cost + output_cost + tool_cost

        costs.append(round(total_cost, 6))

    return costs

# Add cost column to results
evaluation_results['estimated_cost'] = estimate_cost(evaluation_results)

# Calculate cost metrics
cost_summary = evaluation_results.groupby('agent').agg(
    total_cost=('estimated_cost', 'sum'),
    avg_cost_per_task=('estimated_cost', 'mean')
)

# Cost per successful task (accuracy == 1 counts as a success)
successful_counts = evaluation_results.groupby('agent')['accuracy'].sum()
successful_cost = evaluation_results[evaluation_results['accuracy'] == 1].groupby('agent')['estimated_cost'].sum()
cost_summary['cost_per_success'] = (successful_cost / successful_counts).fillna(float('inf'))

print("\nCost Analysis:")
print(cost_summary.to_string())



Cost Analysis:
               total_cost  avg_cost_per_task  cost_per_success
agent                                                         
basic            0.003795           0.003795          0.003795
comprehensive    0.001525           0.001525          0.001525
structured       0.003795           0.003795          0.003795


## Part 10: Generate Recommendations

Generate actionable recommendations based on evaluation results.

In [85]:
def generate_recommendations(agent_summary, cost_summary):
    """Generate recommendations based on evaluation results"""

    prompt_score  = (agent_summary['accuracy'] + agent_summary['completeness'] + agent_summary['clarity']) / 3
    tool_score    = (agent_summary['tool_precision'] + agent_summary['tool_recall']) / 2
    overall_score = (prompt_score + tool_score) / 2

    best_agent = overall_score.idxmax()
    best_score = overall_score.max()

    print("=" * 60)
    print("EVALUATION RECOMMENDATIONS")
    print("=" * 60)
    print(f"  Best overall : {best_agent} (score={best_score:.2f})")
    print(f"  Fastest      : {agent_summary['execution_time'].idxmin()}")

    print("\nPer-Agent:")
    for agent in agent_summary.index:
        row = agent_summary.loc[agent]
        print(f"  {agent.upper()} | score={overall_score[agent]:.2f}")
        print(f"    accuracy={row['accuracy']:.0f}  completeness={row['completeness']:.0f}  recall={row['tool_recall']:.0f}")
        if row['accuracy']    < 0.8: print("    ⚠ Low accuracy")
        if row['completeness']< 0.7: print("    ⚠ Completeness needs improvement")
        if row['tool_recall'] < 0.7: print("    ⚠ Missing expected tools")

    print()
    if best_score >= 0.8:
        print(f"  ✅ '{best_agent}' READY FOR PRODUCTION")
    else:
        print(f"  🔧 Needs refinement (best={best_score:.2f} < 0.8)")
    print("=" * 60)

generate_recommendations(agent_summary, cost_summary)


EVALUATION RECOMMENDATIONS
  Best overall : basic (score=0.83)
  Fastest      : basic

Per-Agent:
  BASIC | score=0.83
    accuracy=1  completeness=1  recall=1
  COMPREHENSIVE | score=0.83
    accuracy=1  completeness=0  recall=1
    ⚠ Completeness needs improvement
  STRUCTURED | score=0.83
    accuracy=1  completeness=1  recall=1

  ✅ 'basic' READY FOR PRODUCTION
